# Formal Speech → Gen Z Meme Language
## LSTM Seq2Seq Baseline + FLAN-T5 QLoRA Fine-Tuning

**Author:** Divya Gunjan

---

Two approaches to translating formal English into Gen Z slang:

1. **LSTM Seq2Seq Baseline** — custom encoder–decoder with teacher forcing
2. **FLAN-T5 + QLoRA** — 4-bit quantized fine-tuning of `google/flan-t5-large`

| Metric | LSTM Baseline | FLAN-T5 QLoRA |
|--------|:---:|:---:|
| BLEU | 42 | **58** |
| ROUGE-L | 0.54 | **0.76** |


---
# Part 1 · LSTM Seq2Seq Baseline

## Data Preparation

Install libraries and set a fixed seed (42) so results are reproducible.


In [ ]:
!pip install datasets
!pip install -U "transformers>=4.40"
!pip -q install transformers datasets peft accelerate bitsandbytes evaluate sacrebleu sentencepiece

In [ ]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer
import torch, random, numpy as np

SEED = 42
torch.manual_seed(SEED);  np.random.seed(SEED);  random.seed(SEED)

### Dataset

We use the **GenZ Slang Dataset** ([MLBtrio/genz-slang-dataset](https://huggingface.co/datasets/MLBtrio/genz-slang-dataset)) — ~1,800 rows with `Slang`, `Description`, `Example`, and `Context` columns.

For the LSTM, `Description` is the input and `Slang` is the target.


In [ ]:
ds = load_dataset("MLBtrio/genz-slang-dataset")   # ≈1,800 rows, single split
print(ds)
ds['train'].shuffle(seed=SEED).select(range(5)).to_pandas()

### Train / Validation / Test Split — 70 / 15 / 15


In [ ]:
from sklearn.model_selection import train_test_split

ds_full = load_dataset("MLBtrio/genz-slang-dataset", split="train")
pairs = [
    {"Description": r["Description"], "Slang": r["Slang"]}
    for r in ds_full
]

# 70/15/15 split
train_pairs, test_pairs = train_test_split(pairs, test_size=0.15, random_state=SEED)
train_pairs, val_pairs  = train_test_split(train_pairs, test_size=0.1765, random_state=SEED)

ds = DatasetDict({
    "train":      train_pairs,
    "validation": val_pairs,
    "test":       test_pairs,
})
print(ds)

### Custom BPE Tokenizer

We train a small BPE tokenizer (vocab = 8,000) on the training corpus so it
handles slang vocabulary efficiently.


In [ ]:
from itertools import chain

def iter_text():
    for r in ds["train"]:
        yield r["Description"] + " </s> " + r["Slang"]   # source + target

tok = AutoTokenizer.from_pretrained("bert-base-uncased")  # starting checkpoint
tok = tok.train_new_from_iterator(iter_text(), vocab_size=8000)

tok.save_pretrained("./slang_tokenizer")
print("Vocab size =", tok.vocab_size)

### Encoding

`encode` converts every example into the tensors the LSTM needs:
- `input_ids` — tokenised source, padded to `MAX_LEN`
- `attention_mask` — marks real tokens vs. padding
- `labels` — tokenised target (no special tokens)


In [ ]:
from datasets import Dataset

MAX_LEN = 25
PAD_ID  = tok.pad_token_id

def encode(batch):
    src_enc = tok(
        batch["Description"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )
    tgt_ids = tok(
        batch["Slang"],
        add_special_tokens=False,
    )["input_ids"]

    batch["input_ids"]      = src_enc["input_ids"]
    batch["attention_mask"] = src_enc["attention_mask"]
    batch["labels"]         = tgt_ids
    return batch

# Convert lists → HF Datasets and apply encoding
train_ds = Dataset.from_list(train_pairs).map(encode, remove_columns=["Description", "Slang"])
val_ds   = Dataset.from_list(val_pairs).map(encode,   remove_columns=["Description", "Slang"])
test_ds  = Dataset.from_list(test_pairs).map(encode,  remove_columns=["Description", "Slang"])

train_ds.set_format(type="torch")
val_ds.set_format(type="torch")
test_ds.set_format(type="torch")

## Model — Encoder–Decoder LSTM (no attention)

A vanilla seq2seq LSTM with teacher forcing (50% probability).
The encoder reads the full source sequence; the decoder generates one token at a time.


In [ ]:
import torch.nn as nn

class SimpleSeq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim=256, hid_dim=512, num_layers=1, pad_idx=PAD_ID):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.encoder   = nn.LSTM(emb_dim, hid_dim, num_layers, batch_first=True)
        self.decoder   = nn.LSTM(emb_dim, hid_dim, num_layers, batch_first=True)
        self.fc_out    = nn.Linear(hid_dim, vocab_size)

    def forward(self, src, src_mask=None, tgt=None, teacher_forcing_ratio=0.5):
        # a) Encode source
        _, (hidden, cell) = self.encoder(self.embedding(src))
        batch_size = src.size(0)
        # b) Start decoder with PAD token
        input_tok  = torch.full((batch_size, 1), PAD_ID, device=src.device)
        max_len    = tgt.size(1) if tgt is not None else MAX_LEN
        outputs    = []

        for t in range(max_len):
            emb_t               = self.embedding(input_tok)
            out, (hidden, cell) = self.decoder(emb_t, (hidden, cell))
            logits              = self.fc_out(out.squeeze(1))
            outputs.append(logits.unsqueeze(1))
            # Teacher forcing vs. greedy argmax
            if tgt is not None and random.random() < teacher_forcing_ratio:
                input_tok = tgt[:, t].unsqueeze(1)
            else:
                input_tok = logits.argmax(1).unsqueeze(1)

        return torch.cat(outputs, dim=1)   # [B, T, vocab_size]

## DataLoaders, Optimizer & Loss


In [ ]:
def collate_fn(batch):
    src_seqs = [ex["input_ids"] for ex in batch]
    tgt_seqs = [ex["labels"]    for ex in batch]
    src = nn.utils.rnn.pad_sequence(src_seqs, batch_first=True, padding_value=PAD_ID)
    tgt = nn.utils.rnn.pad_sequence(tgt_seqs, batch_first=True, padding_value=PAD_ID)
    return {"input_ids": src, "labels": tgt}

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, collate_fn=collate_fn)

In [ ]:
from torch.optim import AdamW
from transformers import get_scheduler

model     = SimpleSeq2Seq(tok.vocab_size).to("cuda")
optimizer = AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

num_epochs = 10
num_steps  = num_epochs * len(train_loader)
lr_sched   = get_scheduler("linear", optimizer=optimizer,
                            num_warmup_steps=0, num_training_steps=num_steps)

## Training Loop


In [ ]:
for epoch in range(num_epochs):
    # — Train
    model.train()
    total_train_loss = 0
    for batch in train_loader:
        src = batch["input_ids"].cuda()
        tgt = batch["labels"].cuda()
        optimizer.zero_grad()
        logits = model(src, tgt=tgt)
        loss   = criterion(logits.view(-1, tok.vocab_size), tgt.view(-1))
        loss.backward()
        optimizer.step()
        lr_sched.step()
        total_train_loss += loss.item()
    avg_train = total_train_loss / len(train_loader)

    # — Validate
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            src    = batch["input_ids"].cuda()
            tgt    = batch["labels"].cuda()
            logits = model(src, tgt=tgt, teacher_forcing_ratio=0.0)
            loss   = criterion(logits.view(-1, tok.vocab_size), tgt.view(-1))
            total_val_loss += loss.item()
    avg_val = total_val_loss / len(val_loader)

    print(f"Epoch {epoch+1} -> train_loss: {avg_train:.3f}, val_loss: {avg_val:.3f}")

## Inference


In [ ]:
def infer_slang(sentence):
    model.eval()
    tok_ids = tok(sentence, return_tensors="pt",
                  padding=True, truncation=True,
                  max_length=MAX_LEN)["input_ids"].cuda()
    with torch.no_grad():
        out = model(tok_ids, tgt=None, teacher_forcing_ratio=0.0)
    pred_id = out.argmax(-1)[0, 0].item()
    return tok.decode([pred_id])

## Evaluation

**BLEU** measures n-gram precision (1–4 grams) with a brevity penalty.
**ROUGE-L** measures the longest common subsequence — captures order and structure.


In [ ]:
import evaluate

model.eval()
preds, refs = [], []

with torch.no_grad():
    for batch in test_loader:
        src      = batch["input_ids"].cuda()
        out      = model(src, tgt=batch["labels"].cuda(), teacher_forcing_ratio=0.0)
        pred_ids = out.argmax(-1).cpu()

        for seq in pred_ids:
            toks = seq.tolist()
            if PAD_ID in toks:
                toks = toks[:toks.index(PAD_ID)]
            preds.append(tok.decode(toks, skip_special_tokens=True))

        for seq in batch["labels"].cpu():
            toks = seq.tolist()
            if PAD_ID in toks:
                toks = toks[:toks.index(PAD_ID)]
            refs.append(tok.decode(toks, skip_special_tokens=True))

bleu       = evaluate.load("sacrebleu")
bleu_score = bleu.compute(predictions=preds, references=[[r] for r in refs])["score"]
print(f"BLEU score: {bleu_score:.2f}")

for idx in random.sample(range(len(preds)), 5):
    print(f"--- Example {idx} ---")
    print("Description   :", test_pairs[idx]["Description"])
    print("Reference Slang:", refs[idx])
    print("Predicted Slang:", preds[idx])
    print()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Analysis — Why LSTM Falls Short

The vanilla Seq2Seq LSTM struggles because it must compress the entire source into a **single fixed hidden vector** with no attention.
Additional bottlenecks: small capacity (1 layer), fixed 50% teacher-forcing, and greedy decoding.

**Solution →** Fine-tune a pretrained transformer (FLAN-T5) with QLoRA,
which has built-in multi-head attention and learns alignment automatically.

---
# Part 2 · FLAN-T5 + QLoRA Fine-Tuning


## Setup


In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate evaluate sentencepiece
!pip install sacrebleu
!pip install rouge_score

In [ ]:
import torch
import random
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import evaluate

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## Load Dataset


In [ ]:
dataset = load_dataset('MLBtrio/genz-slang-dataset', split='train')
print(dataset[0])
print(dataset.column_names)

## Data Preprocessing

We build `(formal, slang)` sentence pairs by substituting the slang token in each example
sentence with its plain-English description.

**Example:**

| Field | Value |
|---|---|
| Slang | `W` |
| Description | `Shorthand for win` |
| Example | `Got the job today, big W!` |
| → formal | `got the job today, big win!` |
| → slang (target) | `Got the job today, big W!` |

Small preprocessing patches applied:
- Strip `"Shorthand for "` prefix from descriptions
- Pick the first meaning when multiple exist (e.g. `"loss/losing"` → `"loss"`)
- Fix article agreement (`"an win"` → `"a win"`)


In [ ]:
def create_pair(entry):
    desc = entry['Description']

    # Strip "Shorthand for " prefix
    prefix = "Shorthand for "
    if isinstance(desc, str) and desc.startswith(prefix):
        meaning = desc[len(prefix):]
    else:
        meaning = desc

    # Pick first meaning when multiple options given (e.g. "loss/losing")
    meaning = meaning.split("/")[0].strip()

    # Replace slang token in the example sentence with plain meaning
    formal = entry["Example"].replace(entry["Slang"], meaning)

    # Fix article: "an loss" -> "a loss"
    formal = formal.replace(f"an {meaning}", f"a {meaning}")

    return {
        "formal": formal.lower().strip(),
        "slang":  entry["Example"],
    }

pairs = [create_pair(e) for e in dataset]
print(pairs[:2])

In [ ]:
from sklearn.model_selection import train_test_split

train_pairs, test_pairs = train_test_split(pairs, test_size=0.15, random_state=42)
train_pairs, val_pairs  = train_test_split(train_pairs, test_size=0.1765, random_state=42)

print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}, Test: {len(test_pairs)}")

## Load FLAN-T5-Large in 4-bit

4-bit NF4 quantisation (via BitsAndBytes) cuts memory ~4× so the 770M-param model fits
on a single T4 GPU.


In [ ]:
model_name = "google/flan-t5-large"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # T5 has no separate PAD token

print("FLAN-T5-Large loaded in 4-bit")

## QLoRA Configuration

**LoRA** keeps the pretrained weights frozen and inserts small trainable adapter matrices
into the query (`q`) and value (`v`) projections of every attention block.

With `r=8` and `lora_alpha=16`, only ~0.1% of parameters are updated —
fast training with minimal memory overhead.


In [ ]:
model = prepare_model_for_kbit_training(model)   # freeze base weights

lora_config = LoraConfig(
    r=8,                        # adapter rank
    lora_alpha=16,              # scaling factor
    target_modules=['q', 'v'],  # hook into attention Q & V projections
    lora_dropout=0.05,          # regularisation
    bias='none',
    task_type='SEQ_2_SEQ_LM',
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

## Tokenise & Prepare Dataset

Each formal sentence is prefixed with an instruction prompt so the model knows the task.


In [ ]:
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

def tokenize_fn(examples):
    inputs = [f"Translate to GenZ slang: {text}" for text in examples["formal"]]
    model_inputs = tokenizer(
        inputs,
        max_length=128,
        padding="max_length",
        truncation=True,
    )
    labels = tokenizer(
        examples["slang"],
        max_length=128,
        padding="max_length",
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_list(train_pairs).map(tokenize_fn, batched=True)
val_ds   = Dataset.from_list(val_pairs).map(tokenize_fn, batched=True)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=peft_model,
    label_pad_token_id=-100,   # mask padding from loss computation
)

## Training Loop


In [ ]:
from torch.utils.data import DataLoader
from transformers import get_scheduler
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
peft_model.to(device)

train_ds = train_ds.remove_columns(["formal", "slang"])
val_ds   = val_ds.remove_columns(["formal", "slang"])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  collate_fn=data_collator)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, collate_fn=data_collator)

optimizer          = AdamW(peft_model.parameters(), lr=2e-4)
num_epochs         = 3
num_training_steps = num_epochs * len(train_loader)

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

for epoch in range(num_epochs):
    # — Train
    peft_model.train()
    total_train_loss = 0
    for batch in train_loader:
        batch   = {k: v.to(device) for k, v in batch.items()}
        outputs = peft_model(**batch)
        loss    = outputs.loss
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        total_train_loss += loss.item()
    avg_train = total_train_loss / len(train_loader)

    # — Validate
    peft_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch   = {k: v.to(device) for k, v in batch.items()}
            outputs = peft_model(**batch)
            total_val_loss += outputs.loss.item()
    avg_val = total_val_loss / len(val_loader)

    print(f"Epoch {epoch+1} -> train loss: {avg_train:.3f}, val loss: {avg_val:.3f}")

## Inference


In [ ]:
def translate_to_slang(texts, max_new_tokens=50):
    prompts = [f"Translate to GenZ slang: {t}" for t in texts]
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128,
    ).to(peft_model.device)

    outs = peft_model.generate(
        input_ids      = enc.input_ids,
        attention_mask = enc.attention_mask,
        max_new_tokens = max_new_tokens,
        do_sample      = False,
    )
    return [tokenizer.decode(o, skip_special_tokens=True) for o in outs]

## Predict on Test Set


In [ ]:
formal_texts    = [ex["formal"] for ex in test_pairs]
reference_texts = [ex["slang"]  for ex in test_pairs]

preds = translate_to_slang(formal_texts)

In [ ]:
indices = random.sample(range(len(formal_texts)), k=5)
for idx in indices:
    print(f"--- Example {idx} ---")
    print("Formal   :", formal_texts[idx])
    print("Reference:", reference_texts[idx])
    print("Predicted:", preds[idx])
    print()

## Evaluation — BLEU & ROUGE-L


In [ ]:
from evaluate import load

# BLEU
bleu      = load("sacrebleu")
bleu_refs = [[r] for r in reference_texts]
bleu_res  = bleu.compute(predictions=preds, references=bleu_refs)
print(f"BLEU score: {bleu_res['score']:.2f}")

# ROUGE-L
rouge     = load("rouge")
rouge_res = rouge.compute(predictions=preds, references=reference_texts)
print("ROUGE-L:", {k: f"{v:.3f}" for k, v in rouge_res.items()})